In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

True

In [38]:
from utils.seku_crd_corpus_utils import *
import random
from utils.citation_context import *

In [11]:
PAPER_DUMP_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/dump"
MAX_REFERENCES = 32
CITATION_CONTEXT_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/citation_contexts"
TEST_SET_IDS = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/main-paper-set-1.txt"

In [9]:
paper_index = load_paper_index(max_files=4096) # turn off the max-files if you want to load all papers from dataset
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 4095line [00:00, 4112.47line/s]


Loaded 4096 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (201.8 MB in memory)
Total 4096 Main papers


### Smoke test with random papers ###

In [17]:
# one-off scratch dump for a specific paper_id -- uses its own variable names
# (not paper_id/main_paper_data/fname) so it can't collide with the random
# main-paper pipeline below, which reuses those names for the paper actually
# used by the rest of the notebook.
_scratch_paper_id = "2312.15391"
_scratch_paper_data = get_paper_from_s3(_scratch_paper_id)
_scratch_fname = os.path.join(PAPER_DUMP_DIR,f"{_scratch_paper_id}.json")
with open(_scratch_fname,"w") as f:
    json.dump(_scratch_paper_data,f,indent=2)

**Take some samples of main papers**

In [18]:
sample = 50
sample_main_paper_ids = random.sample(all_main_paper_ids, sample)
print(f'Total {len(sample_main_paper_ids)} Main papers')

Total 50 Main papers


**Randomly pick a main paper from samples**

In [19]:
rand_idx = random.randint(0,sample-1)
rand_paper_id = sample_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
print(f'Title: {title}')
main_paper_data = get_paper_from_s3(rand_paper_id)
print("Retrieved main paper data from S3...Extracting titles (not needed) and abstract")
fname = os.path.join(PAPER_DUMP_DIR,f"{rand_paper_id}.json")
with open(fname,"w") as f:
    json.dump(main_paper_data,f,indent=2)

Random Main paper Index:16 [Corpus ID: 2306.15964]
Title: Theoretical study of scalar meson $a_0(1710)$ in the $\eta_c \to
  {\bar{K}}^0K^+\pi^- $ reaction
Retrieved main paper data from S3...Extracting titles (not needed) and abstract


**Reference papers**

In [9]:
in_corpus_refs_ids = main_data["in_corpus_references"] # list of IDs of [C]
nb_all_refs_r = main_data["total_bib_entries"]
# we need to check if these ids truly exists in our Full Paper Data
# our full paper data is in S3 😊, so we will get it from there. 

# for quick computation, we will take few references, not ALL
in_corpus_refs_ids = in_corpus_refs_ids[:min(len(in_corpus_refs_ids),MAX_REFERENCES)]
print(f"We will extract entities from {len(in_corpus_refs_ids)} citations")


We will extract entities from 32 citations


In [10]:
x = CitationContextExtractor(S=2)

In [12]:
target_ref = in_corpus_refs_ids[0]
print(target_ref)
result = x.extract(fname, target_ref)                   # list of contexts
# x.extract_many("paper.json", ["2106.04399", "1312.5602"]) 
print(result)

1203.1669
[{'section': 'Introduction', 'ref_id': '98ffb5afc5c3daed78e2b20366fa857da002ab70', 'sentence_index': 8, 'before': ['Neutrino oscillations – Following a brilliant idea from Chen, the SNO Collaboration perfectly solved the problem of missing solar neutrinos \\nu ^{}_e in 2002 by measuring both \\nu ^{}_e and neutrinos of non-electron flavors.', 'On the other hand, the Super-Kamiokande Collaboration provided strong evidence for the disappearance of atmospheric neutrinos \\nu ^{}_\\mu and \\overline{\\nu }^{}_\\mu in 1998.'], 'center': 'The observation of reactor neutrino disappearance in 2012 by the Daya Bay Collaboration leads to a complete picture of three-flavor neutrino oscillations, involving three mixing angles \\lbrace \\theta ^{}_{12}, \\theta ^{}_{23}, \\theta ^{}_{13}\\rbrace and two neutrino mass-squared differences \\lbrace \\Delta m^2_{21}, \\Delta m^2_{31}\\rbrace with \\Delta m^2_{ji} \\equiv m^2_j - m^2_i for ji = 21, 31.The discovery of neutrino oscillations was

In [14]:
result[0]["center"]

'The observation of reactor neutrino disappearance in 2012 by the Daya Bay Collaboration leads to a complete picture of three-flavor neutrino oscillations, involving three mixing angles \\lbrace \\theta ^{}_{12}, \\theta ^{}_{23}, \\theta ^{}_{13}\\rbrace and two neutrino mass-squared differences \\lbrace \\Delta m^2_{21}, \\Delta m^2_{31}\\rbrace with \\Delta m^2_{ji} \\equiv m^2_j - m^2_i for ji = 21, 31.The discovery of neutrino oscillations was awarded the Nobel Prize in Physics in 2015, and the most important experimental collaborations for neutrino oscillations and their leaders shared the Breakthrough Prize in Fundamental Physics in 2016.'

### Run for a set of main papers ###

In [39]:
# collecting the set
def read_paper_ids(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

test_paper_ids = read_paper_ids(TEST_SET_IDS)
print(f"Total {len(test_paper_ids)} main papers in set")

Total 59 main papers in set


In [40]:
# declare a citation context extractor
extractor = CitationContextExtractor(S=1)

In [42]:
status = find_citation_context(test_paper_ids[1],
                      bulk_paper_index=paper_index,
                      citation_context_extractor=extractor,
                      output_dir=CITATION_CONTEXT_DIR)
print(status)

True
